In [ ]:
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
from mask_dataset import WheatBinaryDataset

In [ ]:
DEVICE = "cuda"
EPOCHS = 80
BATCH_SIZE = 16
LEARNING_RATE = 1e-4

# Img Transformation Config
transformation_fn = A.Compose([
    A.Normalize(),
    ToTensorV2(),
])


In [ ]:
train_dataset = WheatBinaryDataset('masks/train', transform=transformation_fn)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

val_dataset = WheatBinaryDataset('masks/valid', transform=transformation_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=4, pin_memory=True)

test_dataset = WheatBinaryDataset('masks/test', transform=transformation_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=4)



In [ ]:
model = smp.DeepLabV3Plus(
    encoder_name="mobilenet_v2",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1, # Binary segmentation
).to(DEVICE)


In [ ]:
criterion = smp.losses.DiceLoss(mode='binary')
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

In [ ]:
@torch.no_grad()
def evaluate_model(trained_model, loader, device):
    trained_model.eval()
    metric_iou = 0
    metric_f1 = 0

    for imgs, canopy_masks in loader:
        imgs, canopy_masks = imgs.to(device), canopy_masks.to(device)


        model_output = trained_model(imgs)

        tp, fp, fn, tn = smp.metrics.get_stats(
            model_output,
            canopy_masks.long(),
            mode='binary',
            threshold=0.5
        )

        # IoU and F1
        metric_iou += smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro")
        metric_f1 += smp.metrics.f1_score(tp, fp, fn, tn, reduction="micro")

    avg_iou = metric_iou / len(loader)
    avg_f1 = metric_f1 / len(loader)

    return avg_iou, avg_f1

In [ ]:
best_val_loss = float('inf') # To save the model with the lowest validation loss

#Starts training
for epoch in range(EPOCHS):

    print(f"\n--- Starting Epoch {epoch+1} ---", flush=True)

    model.train()
    train_loss = 0
    for batch_idx, (images, masks) in enumerate(train_loader):
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

        # Print every 5 batches
        if batch_idx % 5 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}", flush=True)

    model.eval()
    val_loss = 0

    with torch.no_grad(): # Disable gradient tracking to save memory
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            output = model(images)

            v_loss = criterion(output, masks)
            val_loss += v_loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    # Update the Learning Rate based on average validation loss
    scheduler.step(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'low_loss_canopy_seg_model.pth')
        print(f"Epoch {epoch+1}: Val Loss Improved to {avg_val_loss:.4f}. Model Saved!", flush=True)
    else:
        print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}", flush=True)

In [ ]:
# Evaluation
val_iou, val_f1 = evaluate_model(model, val_loader, DEVICE)
test_iou, test_f1 = evaluate_model(model, test_loader, DEVICE)

print(f"Validation mIoU: {val_iou:.4f} | Validation F1 (Dice): {val_f1:.4f}")
print(f"Test mIoU: {test_iou:.4f} | Test F1 (Dice): {test_f1:.4f}")

In [ ]:
best_model = smp.DeepLabV3Plus(
    encoder_name="mobilenet_v2",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1, # Binary segmentation
).to(DEVICE)

state_dict = torch.load("low_loss_canopy_seg_model.pth", map_location=DEVICE)
best_model.load_state_dict(state_dict)

In [ ]:
val_iou, val_f1 = evaluate_model(best_model, val_loader, DEVICE)
test_iou, test_f1 = evaluate_model(best_model, test_loader, DEVICE)

print(f"Validation mIoU: {val_iou:.4f} | Validation F1 (Dice): {val_f1:.4f}")
print(f"Test mIoU: {test_iou:.4f} | Test F1 (Dice): {test_f1:.4f}")

In [ ]:
torch.save(best_model, 'models/best_canopy_seg_model.pt')
# Test mIoU: 0.9240 | Test F1 (Dice): 0.9597
print("Model weights saved successfully.")